# Data Modeling and Databases - Query Optimization Exercise (2026)

In [1]:
# Imports
import time
import json
from pathlib import Path

import duckdb
import pandas as pd

In [2]:
# Config
DB_PATH = "/workspace/data/exercise.duckdb"
PROFILE_DIR = Path("/workspace/data/profiles")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Connect to DuckDB
con = duckdb.connect(DB_PATH)
con.sql("PRAGMA threads=1")
print("Connected to", DB_PATH)

Connected to /workspace/data/exercise.duckdb


In [4]:
# Helper methods

def show_plan(sql: str, analyze=False):
    plan_text = con.sql(f"EXPLAIN {'ANALYZE' if analyze else ''} {sql}").fetchone()[1]
    print(plan_text)

def benchmark(sql: str, runs: int = 3, warmup: int = 1) -> float:
    for _ in range(warmup):
        con.sql(sql).fetchall()

    times_ms = []
    for _ in range(runs):
        t0 = time.perf_counter()
        con.sql(sql).fetchall()
        times_ms.append((time.perf_counter() - t0) * 1000)

    return float(sum(times_ms) / len(times_ms))

def report_off_on(off_avg: float, on_avg: float):
    speedup = off_avg / on_avg if on_avg > 0 else float("inf")
    print(f"\nRelative speedup (OFF/ON, avg): {speedup:.2f}x")
    print(f"optimizer_off: {off_avg:.2f} ms")
    print(f"optimizer_on:  {on_avg:.2f} ms")

# Download TPC-H data

In [5]:
# Install and load TPC-H extension
con.sql("INSTALL tpch")
con.sql("LOAD tpch")

SCALE_FACTOR=0.1

# Drop existing TPC-H tables, then regenerate at SF=0.1 for idempotency.
for t in ["lineitem", "orders", "partsupp", "part", "customer", "supplier", "nation", "region"]:
    con.sql(f"DROP TABLE IF EXISTS {t}")

con.sql(f'CALL dbgen(sf={SCALE_FACTOR})')

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ 0 rows  │
└─────────┘

# TPC-H Schema

<img src="tpch-schema.png" alt="TPC-H Schema" width="700"/>

# Join Reordering

## Example 1: TPC-H Query 5

In [6]:
# TPC-H Query 5 
tpc_h_query_5 = """
SELECT
    n_name,
    SUM(l_extendedprice * (1 - l_discount)) AS revenue
FROM
    customer,
    orders,
    lineitem,
    supplier,
    nation,
    region
WHERE
    c_custkey = o_custkey
    AND l_orderkey = o_orderkey
    AND l_suppkey = s_suppkey
    AND c_nationkey = s_nationkey
    AND s_nationkey = n_nationkey
    AND n_regionkey = r_regionkey
    AND r_name = 'ASIA'
    AND o_orderdate >= DATE '1994-01-01'
    AND o_orderdate < DATE '1994-01-01' + INTERVAL '1 year'
GROUP BY
    n_name
ORDER BY
    revenue DESC
"""

In [7]:
# Turn off join-reordering optimization
con.sql("PRAGMA enable_optimizer")
con.sql("SET disabled_optimizers='join_order'")
print("=== EXPLAIN: TPC-H Query 5 with join_order OFF ===")
show_plan(tpc_h_query_5)
off_avg_1 = benchmark(tpc_h_query_5)

=== EXPLAIN: TPC-H Query 5 with join_order OFF ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│     sum((exercise.main    │
│ .lineitem.l_extendedprice │
│    * (1 - exercise.main   │
│  .lineitem.l_discount)))  │
│            DESC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_string_│
│        uhugeint(#0)       │
│             #1            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)  

In [8]:
# Turn on join-reordering optimization 
con.sql("SET disabled_optimizers=''")
print("\n=== EXPLAIN: TPC-H Query 5 with join_order ON ===")
show_plan(tpc_h_query_5)
on_avg_1 = benchmark(tpc_h_query_5)


=== EXPLAIN: TPC-H Query 5 with join_order ON ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│     sum((exercise.main    │
│ .lineitem.l_extendedprice │
│    * (1 - exercise.main   │
│  .lineitem.l_discount)))  │
│            DESC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_string_│
│        uhugeint(#0)       │
│             #1            │
│                           │
│        ~9,169 rows        │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)  

In [9]:
report_off_on(off_avg_1, on_avg_1)


Relative speedup (OFF/ON, avg): 1.11x
optimizer_off: 5.29 ms
optimizer_on:  4.76 ms


## Example 2: TPC-H Query 9


In [10]:
# TPC-H Query 9
tpc_h_query_9 = """
SELECT
    nation,
    o_year,
    SUM(amount) AS sum_profit
FROM (
    SELECT
        n_name AS nation,
        EXTRACT(YEAR FROM o_orderdate) AS o_year,
        l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity AS amount
    FROM
        part,
        supplier,
        lineitem,
        partsupp,
        orders,
        nation
    WHERE
        s_suppkey = l_suppkey
        AND ps_suppkey = l_suppkey
        AND ps_partkey = l_partkey
        AND p_partkey = l_partkey
        AND o_orderkey = l_orderkey
        AND s_nationkey = n_nationkey
        AND p_name LIKE '%green%'
) AS profit
GROUP BY
    nation,
    o_year
ORDER BY
    nation,
    o_year DESC
"""

In [11]:
# OFF
con.sql("PRAGMA enable_optimizer")
con.sql("SET disabled_optimizers='join_order'")
print("=== EXPLAIN: TPC-H Query 9 with join_order OFF ===")
show_plan(tpc_h_query_9)
off_avg_2 = benchmark(tpc_h_query_9)

=== EXPLAIN: TPC-H Query 9 with join_order OFF ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│__internal_decompress_integ│
│    ral_bigint(#1, 1992)   │
│             #2            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│     profit.nation ASC     │
│     profit.o_year DESC    │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_string_│
│        uhugeint(#0)       │
│__internal_compress_integra│
│    l_utinyint(#1, 1992)   │
│             #2            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompr

In [12]:
# ON
con.sql("SET disabled_optimizers=''")
print("\n=== EXPLAIN: TPC-H Query 9 with join_order ON ===")
show_plan(tpc_h_query_9)
on_avg_2 = benchmark(tpc_h_query_9)


=== EXPLAIN: TPC-H Query 9 with join_order ON ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│__internal_decompress_integ│
│    ral_bigint(#1, 1992)   │
│             #2            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│     profit.nation ASC     │
│     profit.o_year DESC    │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_string_│
│        uhugeint(#0)       │
│__internal_compress_integra│
│    l_utinyint(#1, 1992)   │
│             #2            │
│                           │
│       ~375,331 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompr

In [13]:
report_off_on(off_avg_2, on_avg_2)


Relative speedup (OFF/ON, avg): 3.26x
optimizer_off: 37.70 ms
optimizer_on:  11.56 ms


# 2) Filter pushdown

## Example 1

In [14]:
filter_pushdown_q1 = """
SELECT COUNT(*)
FROM orders o
JOIN lineitem l ON l.l_orderkey = o.o_orderkey
JOIN customer c ON c.c_custkey = o.o_custkey
JOIN nation n ON n.n_nationkey = c.c_nationkey
WHERE o.o_orderdate >= DATE '1995-01-01'
  AND o.o_orderdate < DATE '1995-07-01'
  AND l.l_shipmode = 'AIR'
  AND c.c_mktsegment = 'AUTOMOBILE'
"""

In [15]:
# filter_pushdown OFF only
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: join query with filter_pushdown OFF ===")
show_plan(filter_pushdown_q1)
fp1_off_avg = benchmark(filter_pushdown_q1)

=== EXPLAIN: join query with filter_pushdown OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│   ((o_orderdate >= CAST(  │
│ '1995-01-01' AS DATE)) AND│
│  (o_orderdate < CAST('1995│
│   -07-01' AS DATE)) AND   │
│  (l_shipmode = CAST('AIR' │
│      AS VARCHAR)) AND     │
│   (c_mktsegment = CAST(   │
│ 'AUTOMOBILE' AS VARCHAR)))│
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├────────────────────────────────────────────────────────────────────────┐
│ c_nationkey = n_nationkey │                                                                 

In [16]:
# filter_pushdown ON
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: join query with filter_pushdown ON ===")
show_plan(filter_pushdown_q1)
fp1_on_avg = benchmark(filter_pushdown_q1)


=== EXPLAIN: join query with filter_pushdown ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├──────────────┐
│  l_orderkey = o_orderkey  │              │
│                           │              │
│        ~4,790 rows        │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│         SEQ_SCAN          ││         HASH_JOIN         │
│    ────────────────────   ││    ────────────────────   │
│      Table: lineitem      ││      Join Type: INNER     │
│   Type: Sequential Scan   ││                           │
│                           ││        Conditions:        │
│        Projections:       ││   o_cus

In [17]:
report_off_on(fp1_off_avg, fp1_on_avg) 


Relative speedup (OFF/ON, avg): 12.23x
optimizer_off: 38.52 ms
optimizer_on:  3.15 ms


# Expression Rewriter

## Example 1: Intersecting Ranges

In [18]:
query_between_overlap = """
SELECT COUNT(*)
FROM orders
WHERE o_totalprice BETWEEN 1000 AND 5000
  AND o_totalprice BETWEEN 3000 AND 8000
"""

In [19]:
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: overlapping BETWEEN with optimizer OFF ===")
show_plan(query_between_overlap)
between_off_avg = benchmark(query_between_overlap)

=== EXPLAIN: overlapping BETWEEN with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│   ((o_totalprice >= CAST  │
│  (1000 AS DECIMAL(15,2))) │
│  AND (o_totalprice >= CAST│
│  (3000 AS DECIMAL(15,2))) │
│  AND (o_totalprice <= CAST│
│  (5000 AS DECIMAL(15,2))) │
│  AND (o_totalprice <= CAST│
│ (8000 AS DECIMAL(15,2)))) │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│       Table: orders       │
│   Type: Sequential Scan   │
│                           │
│          ~0 rows          │
└───────────────────────────┘



In [20]:
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: overlapping BETWEEN with optimizer ON ===")
show_plan(query_between_overlap)
between_on_avg = benchmark(query_between_overlap)


=== EXPLAIN: overlapping BETWEEN with optimizer ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│       Table: orders       │
│   Type: Sequential Scan   │
│                           │
│          Filters:         │
│ o_totalprice>=3000.00 AND │
│    o_totalprice<=5000.00  │
│                           │
│        ~30,000 rows       │
└───────────────────────────┘



In [21]:
report_off_on(between_off_avg, between_on_avg)


Relative speedup (OFF/ON, avg): 1.47x
optimizer_off: 0.54 ms
optimizer_on:  0.36 ms


## Example 2: IN-Clause Rewrite

In [22]:
values_in = ", ".join(str(i) for i in range(1, 601))
query_in_clause = f"""
SELECT COUNT(*)
FROM orders
WHERE o_custkey IN ({values_in})
"""

In [23]:
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: large IN-clause query with optimizer OFF ===")
show_plan(query_in_clause)
in_off_avg = benchmark(query_in_clause)

=== EXPLAIN: large IN-clause query with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│  (o_custkey IN (CAST(1 AS │
│  BIGINT), CAST(2 AS BIGINT│
│ ), CAST(3 AS BIGINT), CAST│
│  (4 AS BIGINT), CAST(5 AS │
│  BIGINT), CAST(6 AS BIGINT│
│ ), CAST(7 AS BIGINT), CAST│
│  (8 AS BIGINT), CAST(9 AS │
│     BIGINT), CAST(10 AS   │
│     BIGINT), CAST(11 AS   │
│     BIGINT), CAST(12 AS   │
│     BIGINT), CAST(13 AS   │
│     BIGINT), CAST(14 AS   │
│     BIGINT), CAST(15 AS   │
│     BIGINT), CAST(16 AS   │
│     BIGINT), CAST(17 AS   │
│     BIGINT), CAST(18 AS   │
│     BIGINT), CAST(19 AS   │
│     BIGINT), CAST(20 AS   │
│     BIGINT), CAST(21 AS   │
│     BIGINT), CAST(22 AS   │
│     BIGINT), CAST(23 AS   │
│     BIGINT), CAST(24 AS   │
│     BIGINT

In [24]:
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: large IN-clause query with optimizer ON ===")
show_plan(query_in_clause)
in_on_avg = benchmark(query_in_clause)


=== EXPLAIN: large IN-clause query with optimizer ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│       Table: orders       │
│   Type: Sequential Scan   │
│                           │
│          Filters:         │
│ o_custkey>=1 AND o_custkey│
│           <=600           │
│                           │
│        ~30,000 rows       │
└───────────────────────────┘



In [25]:
report_off_on(in_off_avg, in_on_avg)


Relative speedup (OFF/ON, avg): 10.47x
optimizer_off: 21.67 ms
optimizer_on:  2.07 ms


## Example 3: Constant-false Predicates

In [26]:
query_count_impossible = """
SELECT COUNT(*)
FROM orders
WHERE o_orderdate < DATE '1994-01-01'
  AND o_orderdate > DATE '1994-12-31'
"""

In [27]:
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: impossible COUNT query with optimizer OFF ===")
show_plan(query_count_impossible)
count_impossible_off_avg = benchmark(query_count_impossible)

=== EXPLAIN: impossible COUNT query with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ ((o_orderdate < CAST('1994│
│   -01-01' AS DATE)) AND   │
│ (o_orderdate > CAST('1994 │
│     -12-31' AS DATE)))    │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│       Table: orders       │
│   Type: Sequential Scan   │
│                           │
│          ~0 rows          │
└───────────────────────────┘



In [28]:
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: impossible COUNT query with optimizer ON ===")
show_plan(query_count_impossible)
count_impossible_on_avg = benchmark(query_count_impossible)


=== EXPLAIN: impossible COUNT query with optimizer ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        EMPTY_RESULT       │
└───────────────────────────┘



In [29]:
report_off_on(count_impossible_off_avg, count_impossible_on_avg)


Relative speedup (OFF/ON, avg): 2.24x
optimizer_off: 0.28 ms
optimizer_on:  0.12 ms


## Example 4: Constant-false subtree pruning 

In [30]:
query_false_prune = """
SELECT COUNT(*)
FROM orders o
JOIN lineitem l ON l.l_orderkey = o.o_orderkey
JOIN customer c ON c.c_custkey = o.o_custkey
WHERE 1 = 0
"""

In [31]:
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: constant-false query with optimizer OFF ===")
show_plan(query_false_prune)
false_off_avg = benchmark(query_false_prune)

=== EXPLAIN: constant-false query with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ (CAST(1 AS INTEGER) = CAST│
│      (0 AS INTEGER))      │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├───────────────────────────────────────────┐
│   o_custkey = c_custkey   │                                           │
│                           │                                           │
│          ~0 rows          │                                           │
└─────────────┬─────────────┘                                           │
┌──

In [32]:
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: constant-false query with optimizer ON ===")
show_plan(query_false_prune)
false_on_avg = benchmark(query_false_prune)


=== EXPLAIN: constant-false query with optimizer ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        EMPTY_RESULT       │
└───────────────────────────┘



In [33]:
report_off_on(false_off_avg, false_on_avg)


Relative speedup (OFF/ON, avg): 168.08x
optimizer_off: 21.70 ms
optimizer_on:  0.13 ms


## Example 5: Constant-true Predicates

In [34]:
query_expr_raw = """
SELECT COUNT(*)
FROM orders o, customer c
WHERE ((o.o_orderkey + 0) = 345678)
  AND ((o.o_custkey * 1) > 0)
  AND ((o.o_orderstatus = 'O') OR TRUE)
  AND ((c.c_comment IS NULL) OR (c.c_comment IS NOT NULL))
  AND (10 - 10 = 0)
  AND TRUE
"""

In [35]:
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: raw expression-heavy query with optimizer OFF ===")
show_plan(query_expr_raw)
expr_off_avg = benchmark(query_expr_raw)

=== EXPLAIN: raw expression-heavy query with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ (((o_orderkey + CAST(0 AS │
│  BIGINT)) = CAST(345678 AS│
│  BIGINT)) AND ((o_custkey │
│   * CAST(1 AS BIGINT)) >  │
│  CAST(0 AS BIGINT)) AND ( │
│ (o_orderstatus = CAST('O' │
│  AS VARCHAR)) OR CAST('t' │
│     AS BOOLEAN)) AND (    │
│  (c_comment IS NULL) OR   │
│  (c_comment IS NOT NULL)) │
│   AND ((10 - 10) = CAST(0 │
│  AS INTEGER)) AND CAST('t'│
│        AS BOOLEAN))       │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       CROSS_PRODUCT       ├──────────────┐
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│       

In [36]:
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: raw expression-heavy query with optimizer ON ===")
show_plan(query_expr_raw)
expr_on_avg = benchmark(query_expr_raw)


=== EXPLAIN: raw expression-heavy query with optimizer ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       CROSS_PRODUCT       ├──────────────┐
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│         SEQ_SCAN          ││         SEQ_SCAN          │
│    ────────────────────   ││    ────────────────────   │
│      Table: customer      ││       Table: orders       │
│   Type: Sequential Scan   ││   Type: Sequential Scan   │
│                           ││                           │
│                           ││          Filters:         │
│                           ││     o_orderkey=345678     │
│                           ││                           │
│        ~3,000 rows        ││          ~2 rows          │
└───────────────────────────┘└─────────────────

In [37]:
report_off_on(expr_off_avg, expr_on_avg)


Relative speedup (OFF/ON, avg): 12976.26x
optimizer_off: 5485.71 ms
optimizer_on:  0.42 ms


# TOP-N query optimization

## Example 1

In [38]:
query_topn = """
SELECT
    l_orderkey,
    l_partkey,
    l_extendedprice,
    l_quantity
FROM lineitem
ORDER BY l_extendedprice
LIMIT 20
"""

In [39]:
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: Top-N query with optimizer OFF ===")
show_plan(query_topn)
topn_off_avg = benchmark(query_topn)

=== EXPLAIN: Top-N query with optimizer OFF ===
┌───────────────────────────┐
│      STREAMING_LIMIT      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│   exercise.main.lineitem  │
│    .l_extendedprice ASC   │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│      Table: lineitem      │
│   Type: Sequential Scan   │
│                           │
│          ~0 rows          │
└───────────────────────────┘



In [40]:
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: Top-N query with optimizer ON ===")
show_plan(query_topn)
topn_on_avg = benchmark(query_topn)


=== EXPLAIN: Top-N query with optimizer ON ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_integ│
│     ral_bigint(#0, 1)     │
│__internal_decompress_integ│
│     ral_bigint(#1, 1)     │
│             #2            │
│             #3            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│   exercise.main.lineitem  │
│    .l_extendedprice ASC   │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_integra│
│     l_uinteger(#0, 1)     │
│__internal_compress_integra│
│     l_usmallint(#1, 1)    │
│             #2            │
│             #3            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN   

In [41]:
report_off_on(topn_off_avg, topn_on_avg)


Relative speedup (OFF/ON, avg): 18.65x
optimizer_off: 32.10 ms
optimizer_on:  1.72 ms


# Query Optimizers are NOT perfect!

In [42]:
query_not_perfect_union = """
SELECT COUNT(*)
FROM (
    SELECT o_orderkey
    FROM orders
    WHERE o_orderstatus = 'F'

    UNION

    SELECT o_orderkey
    FROM orders
    WHERE o_orderstatus = 'O'

    UNION

    SELECT o_orderkey
    FROM orders
    WHERE o_orderstatus = 'P'
) u
"""

In [43]:
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: Unions with optimizer OFF ===")
show_plan(query_not_perfect_union)
not_perfect_union_off_avg = benchmark(query_not_perfect_union)

=== EXPLAIN: Unions with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           UNION           ├───────────────────────────────────────────┐
└─────────────┬─────────────┘                                           │
┌─────────────┴─────────────┐                             ┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │                             │         PROJECTION        │
│    ────────────────────   │                             │    ────────────────────   │
│         Groups: #0        │                             │         o_orderkey        │
│                           │        

In [44]:
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: Unions with optimizer ON ===")
show_plan(query_not_perfect_union)
not_perfect_union_on_avg = benchmark(query_not_perfect_union)


=== EXPLAIN: Unions with optimizer ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        ~75,000 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           UNION           ├───────────────────────────────────────────┐
└─────────────┬─────────────┘                                           │
┌─────────────┴─────────────┐                             ┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │                             │         SEQ_SCAN          │
│    ────────────────────   │                             │    ────────────────────   │
│         Groups: #0        │                             │       Table: orders       │
│                           │        

In [45]:
report_off_on(not_perfect_union_off_avg, not_perfect_union_on_avg)


Relative speedup (OFF/ON, avg): 1.11x
optimizer_off: 10.09 ms
optimizer_on:  9.11 ms
